In [1]:
import pandas as pd 
import numpy as np

In [2]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

##### Reading the data set and nameing the columns and removing the unwanted column timestamp

In [3]:
df=pd.read_csv("Ratings_.csv",names=['user_id','prod_id','ratings','timestamp'])
df.drop('timestamp',axis=1,inplace=True)
df.head()
df.shape

(7824482, 3)

In [4]:
##### preprocessing the data

In [5]:
df.isna().sum()

user_id    0
prod_id    0
ratings    0
dtype: int64

In [6]:
# Count user interactions
user_counts = df['user_id'].value_counts()

# Users with 5+ interactions
active_users = user_counts[
    user_counts >= 5
].index

# Update df with active users only
df = df[df['user_id'].isin(active_users)]

In [7]:
# Count product interactions
product_counts = df['prod_id'].value_counts()

# Products with 5+ interactions
popular_products = product_counts[
    product_counts >= 5
].index

# Update df with popular products only
df = df[df['prod_id'].isin(popular_products)]

In [8]:
##unique active users
unique_active_users = pd.DataFrame({
    'user_id': df['user_id'].unique()
})

In [9]:
## unique popular produts
unique_popular_products = pd.DataFrame({
    'prod_id': df['prod_id'].unique()
})


In [10]:
df.shape

(1936886, 3)

In [11]:
from scipy.sparse import csr_matrix

In [12]:
# User index
df['user_index'] = df['user_id'].astype(
    'category'
).cat.codes

# Product index
df['product_index'] = df['prod_id'].astype(
    'category'
).cat.codes

print("\nIndexes Created!")

print(df.head())


Indexes Created!
           user_id     prod_id  ratings  user_index  product_index
13   AO94DHGC771SJ  0528881469      5.0      232138              0
14   AMO214LNFCEI4  0528881469      1.0      229131              0
16  A3N7T0DY83Y4IG  0528881469      3.0      177409              0
17  A1H8PY3QHMQQA0  0528881469      2.0       32428              0
35  A24EV6RXELQZ63  0528881469      1.0       75879              0


In [13]:
## creating the sparse matrxis
sparse_matrix = csr_matrix(

    (
        df['ratings'],

        (
            df['product_index'],
            df['user_index']
        )

    )

)

In [14]:
### Train knn
from sklearn.neighbors import NearestNeighbors
model_knn = NearestNeighbors(

    metric='cosine',
    algorithm='brute',
    n_neighbors=10,
    n_jobs=-1

)

model_knn.fit(sparse_matrix)

,n_neighbors,10
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [15]:
### product mapping
product_to_index = dict(

    zip(
        df['prod_id'],
        df['product_index']
    )

)

index_to_product = dict(

    zip(
        df['product_index'],
        df['prod_id']
    )

)

In [16]:
## Similar product function
def get_similar_products(product_id, top_n=10):

    if product_id not in product_to_index:
        return []

    product_idx = product_to_index[
        product_id
    ]

    distances, indices = model_knn.kneighbors(

        sparse_matrix[product_idx],

        n_neighbors=top_n + 1

    )

    similar_products = []

    for i in range(1, len(indices.flatten())):

        idx = indices.flatten()[i]

        similar_product = index_to_product[idx]

        similarity_score = 1 - distances.flatten()[i]

        similar_products.append([

            similar_product,
            similarity_score

        ])

    return pd.DataFrame(

        similar_products,

        columns=[
            'prod_id',
            'similarity_score'
        ]

    )

In [17]:
## collect candidate product
def generate_candidates(product_id):

    # Similar products
    similar_products_df = get_similar_products(
        product_id,
        top_n=10
    )

    similar_products = similar_products_df[
        'prod_id'
    ].values

    # Users who liked selected product
    users = get_top_users(product_id)

    # Products interacted by those users
    candidate_df = df[
        df['user_id'].isin(users)
    ]

    # Remove original product
    candidate_df = candidate_df[
        candidate_df['prod_id'] != product_id
    ]

    # Keep similar products only
    candidate_df = candidate_df[
        candidate_df['prod_id'].isin(
            similar_products
        )
    ]

    return candidate_df

In [18]:
## feature engineering
product_avg_rating = df.groupby(
    'prod_id'
)['ratings'].mean()

product_count = df.groupby(
    'prod_id'
)['ratings'].count()

user_avg_rating = df.groupby(
    'user_id'
)['ratings'].mean()

user_count = df.groupby(
    'user_id'
)['ratings'].count()

In [19]:
### creating trainning data
rows = []

sample_products = df['prod_id'].unique()[:1000]

In [20]:
def get_top_users(product_id):

    users = df[

        (df['prod_id'] == product_id) &
        (df['ratings'] >= 4)

    ]['user_id'].unique()

    return users

In [21]:
def generate_candidates(product_id):

    # Users who liked selected product
    users = get_top_users(product_id)

    # Products interacted by those users
    candidate_df = df[
        df['user_id'].isin(users)
    ]

    # Remove original product
    candidate_df = candidate_df[
        candidate_df['prod_id'] != product_id
    ]

    return candidate_df

In [22]:
# =========================================================
# TRAINING LOOP
# =========================================================

rows = []

sample_products = df['prod_id'].unique()[:1000]

for product_id in sample_products:

    try:

        # Generate candidate products
        candidate_df = generate_candidates(product_id)

        # Skip if empty
        if candidate_df.empty:
            continue

        # Get similar products once
        similarity_df = get_similar_products(
            product_id,
            top_n=10
        )

        # Loop through candidate products
        for _, row_data in candidate_df.iterrows():

            user_id = row_data['user_id']
            candidate_product = row_data['prod_id']
            rating = row_data['ratings']

            # ====================================
            # SIMILARITY SCORE
            # ====================================

            similarity_row = similarity_df[
                similarity_df['prod_id'] == candidate_product
            ]

            if len(similarity_row) > 0:

                similarity_score = similarity_row[
                    'similarity_score'
                ].values[0]

            else:

                similarity_score = 0

            # ====================================
            # PRODUCT FEATURES
            # ====================================

            avg_product_rating = product_avg_rating.get(
                candidate_product,
                0
            )

            total_product_interactions = product_count.get(
                candidate_product,
                0
            )

            # ====================================
            # USER FEATURES
            # ====================================

            avg_user_rating = user_avg_rating.get(
                user_id,
                0
            )

            total_user_interactions = user_count.get(
                user_id,
                0
            )

            # ====================================
            # TARGET
            # ====================================

            target = 1 if rating >= 4 else 0

            # ====================================
            # STORE FEATURES
            # ====================================

            rows.append([

                similarity_score,
                avg_product_rating,
                total_product_interactions,
                avg_user_rating,
                total_user_interactions,
                target

            ])

    except Exception as e:

        print("Error in product:", product_id)
        print(e)

# =========================================================
# CREATE TRAINING DATAFRAME
# =========================================================

train_df = pd.DataFrame(

    rows,

    columns=[

        'similarity_score',
        'avg_product_rating',
        'product_interactions',
        'avg_user_rating',
        'user_interactions',
        'target'

    ]

)

print(train_df.head())

print("\nTraining Shape:")
print(train_df.shape)

   similarity_score  avg_product_rating  product_interactions  \
0          0.000000            4.535934                   487   
1          0.352148            3.333333                     9   
2          0.000000            4.629199                   774   
3          0.000000            4.269058                   223   
4          0.182574            4.000000                    17   

   avg_user_rating  user_interactions  target  
0         4.857143                  7       1  
1         4.857143                  7       1  
2         4.857143                  7       1  
3         4.857143                  7       1  
4         4.857143                  7       1  

Training Shape:
(316824, 6)


In [23]:
## training data frame
train_df = pd.DataFrame(

    rows,

    columns=[

        'similarity_score',
        'avg_product_rating',
        'product_interactions',
        'avg_user_rating',
        'user_interactions',
        'target'

    ]

)

print(train_df.head())

print(train_df.shape)

   similarity_score  avg_product_rating  product_interactions  \
0          0.000000            4.535934                   487   
1          0.352148            3.333333                     9   
2          0.000000            4.629199                   774   
3          0.000000            4.269058                   223   
4          0.182574            4.000000                    17   

   avg_user_rating  user_interactions  target  
0         4.857143                  7       1  
1         4.857143                  7       1  
2         4.857143                  7       1  
3         4.857143                  7       1  
4         4.857143                  7       1  
(316824, 6)


In [24]:
train_df['target'].value_counts()

target
1    263409
0     53415
Name: count, dtype: int64

In [25]:
# =========================================================
# FEATURES AND TARGET
# =========================================================

X = train_df.drop(
    'target',
    axis=1
)

y = train_df['target']

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

# =========================================================
# TRAIN RANDOM FOREST
# =========================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(

    n_estimators=100,

    max_depth=10,

    random_state=42,

    n_jobs=-1

)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest Model Trained!")

Random Forest Model Trained!


In [26]:
## Evaluateing the model
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Prediction
y_pred = rf_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:")
print(accuracy)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy:
0.8576659038901602

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.29      0.41     10741
           1       0.87      0.97      0.92     52624

    accuracy                           0.86     63365
   macro avg       0.78      0.63      0.67     63365
weighted avg       0.84      0.86      0.83     63365



In [27]:
## recommendation
# =========================================================
# FINAL HYBRID RECOMMENDATION FUNCTION
# =========================================================

def hybrid_recommendation(product_id, top_n=10):

    # =========================================
    # GENERATE CANDIDATES
    # =========================================

    candidate_df = generate_candidates(
        product_id
    )

    # Empty check
    if candidate_df.empty:

        return "No Recommendations Found"

    # =========================================
    # SIMILAR PRODUCTS
    # =========================================

    similarity_df = get_similar_products(
        product_id,
        top_n=20
    )

    recommendation_rows = []

    # =========================================
    # LOOP THROUGH CANDIDATES
    # =========================================

    for _, row in candidate_df.iterrows():

        user_id = row['user_id']

        candidate_product = row['prod_id']

        # =====================================
        # SIMILARITY SCORE
        # =====================================

        similarity_row = similarity_df[

            similarity_df['prod_id']
            == candidate_product

        ]

        if len(similarity_row) > 0:

            similarity_score = similarity_row[
                'similarity_score'
            ].values[0]

        else:

            similarity_score = 0

        # =====================================
        # PRODUCT FEATURES
        # =====================================

        avg_product_rating = product_avg_rating.get(
            candidate_product,
            0
        )

        total_product_interactions = product_count.get(
            candidate_product,
            0
        )

        # =====================================
        # USER FEATURES
        # =====================================

        avg_user_rating = user_avg_rating.get(
            user_id,
            0
        )

        total_user_interactions = user_count.get(
            user_id,
            0
        )

        # =====================================
        # CREATE FEATURE DATAFRAME
        # =====================================

        features = pd.DataFrame([{

            'similarity_score': similarity_score,

            'avg_product_rating': avg_product_rating,

            'product_interactions': total_product_interactions,

            'avg_user_rating': avg_user_rating,

            'user_interactions': total_user_interactions

        }])

        # =====================================
        # RANDOM FOREST SCORE
        # =====================================

        score = rf_model.predict_proba(
            features
        )[0][1]

        recommendation_rows.append([

            candidate_product,
            score

        ])

    # =========================================
    # CREATE RECOMMENDATION DATAFRAME
    # =========================================

    recommendation_df = pd.DataFrame(

        recommendation_rows,

        columns=[

            'recommended_product',
            'score'

        ]

    )

    # Remove duplicates
    recommendation_df = recommendation_df.groupby(

        'recommended_product'

    )['score'].mean().reset_index()

    # Sort scores
    recommendation_df = recommendation_df.sort_values(

        by='score',

        ascending=False

    )

    # Return top products
    return recommendation_df.head(top_n)

In [29]:
# =========================================================
# TEST RECOMMENDATION SYSTEM
# =========================================================

sample_product = df['prod_id'].iloc[0]

print("Selected Product:")
print(sample_product)

recommendations = hybrid_recommendation(

    sample_product,

    top_n=10

)

print("\nTop Recommendations:")

print(recommendations)

Selected Product:
0528881469

Top Recommendations:
  recommended_product     score
4          B0096TK6MI  0.997918
5          B00DUKJ5CQ  0.997227
2          B003ZBZ64Q  0.992504
0          B0013G8PTS  0.991653
1          B001TQSFXS  0.984969
3          B0075SUHKI  0.963582


In [30]:
import pickle

# Save Random Forest model
pickle.dump(
    rf_model,
    open('rf_model.pkl', 'wb')
)

# Save KNN model
pickle.dump(
    model_knn,
    open('knn_model.pkl', 'wb')
)
# Save mappings
pickle.dump(
    product_to_index,
    open('product_to_index.pkl', 'wb')
)

pickle.dump(
    index_to_product,
    open('index_to_product.pkl', 'wb')
)

# Save dataset
df.to_csv(
    'final_dataset.csv',
    index=False
)